In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "postgres")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
conn = create_engine(connection_string)

## Runtime Requirements for Test Execution per Project + Variant

In [ ]:
import pandas as pd

# Query to get test runtimes
query = """
SELECT 
    t.project_id,
    project_name(t.project_id) AS project_name,
    te.variant,
    te.variant_order,
    te.runtime
FROM test t
JOIN mv_test_extension te ON t.id = te.test_id
"""

# Load data into a DataFrame
df_all_runtimes = pd.read_sql_query(query, conn)

# Count zero runtime tests for each group
zero_runtime_counts = df_all_runtimes[df_all_runtimes['runtime'] == 0].groupby(
    ['project_id', 'project_name', 'variant', 'variant_order']
).size().reset_index(name='zero_runtime_count')

# Group by project and variant, then calculate statistics
test_runtime_stats = df_all_runtimes.groupby(['project_id', 'project_name', 'variant', 'variant_order']).agg(
    test_count=('runtime', 'count'),
    total_runtime=('runtime', 'sum'),
    mean_runtime=('runtime', 'mean'),
    median_runtime=('runtime', 'median'),
    min_runtime=('runtime', 'min'),
    max_runtime=('runtime', 'max'),
).reset_index()

# Merge the zero runtime counts into the statistics DataFrame
test_runtime_stats = pd.merge(
    test_runtime_stats, 
    zero_runtime_counts, 
    on=['project_id', 'project_name', 'variant', 'variant_order'], 
    how='left'
)

# Fill NaN values with 0 (groups with no zero-runtime tests)
test_runtime_stats['zero_runtime_count'] = test_runtime_stats['zero_runtime_count'].fillna(0).astype(int)

# Calculate percentage of tests with zero runtime
test_runtime_stats['zero_runtime_pct'] =(
    (test_runtime_stats['zero_runtime_count'] / test_runtime_stats['test_count'] * 100).round(2))

# Round numeric columns for better readability
numeric_cols = ['mean_runtime', 'median_runtime', 'min_runtime', 'max_runtime', 'total_runtime']
for col in numeric_cols:
    test_runtime_stats[col] = test_runtime_stats[col].round(3)

# Sort by project_name, project_id, and variant_order
test_runtime_stats = test_runtime_stats.sort_values(['project_name', 'project_id', 'variant_order'])

# Display the result
print("Test Runtime Statistics by Project and Variant:")
display(test_runtime_stats.loc[:, test_runtime_stats.columns != 'variant_order'])

## Runtime Requirements for Generalization Execution per Project + Variant

In [ ]:
import pandas as pd

# Query to get generalization runtimes
query = """
SELECT 
    g.project_id,
    project_name(g.project_id) AS project_name,
    ge.variant,
    ge.variant_order,
    ge.runtime
FROM generalization g
JOIN mv_generalization_extension ge ON g.id = ge.generalization_id
"""

# Load data into a DataFrame
df_gen_runtimes = pd.read_sql_query(query, conn)

# Group by project and variant, then calculate statistics
gen_runtime_stats = df_gen_runtimes.groupby(['project_id', 'project_name', 'variant', 'variant_order']).agg(
    generalization_count=('runtime', 'count'),
    total_runtime=('runtime', 'sum'),
    mean_runtime=('runtime', 'mean'),
    median_runtime=('runtime', 'median'),
    min_runtime=('runtime', 'min'),
    max_runtime=('runtime', 'max'),
).reset_index()

# Round numeric columns for better readability
numeric_cols = ['mean_runtime', 'median_runtime', 'min_runtime', 'max_runtime', 'total_runtime']
for col in numeric_cols:
    gen_runtime_stats[col] = gen_runtime_stats[col].round(3)

# Sort by project_name, project_id, and variant_order
gen_runtime_stats = gen_runtime_stats.sort_values(['project_name', 'project_id', 'variant_order'])

# Display the result
print("Generalization Runtime Statistics by Project and Variant:")
display(gen_runtime_stats.loc[:, gen_runtime_stats.columns != 'variant_order'])

## Comparison of Test vs. Generalization Runtimes

In [ ]:
query = """
SELECT * FROM mv_runtime_comparison_test_vs_generalization
"""

df = pd.read_sql_query(query, conn)
display(df.loc[:, df.columns != 'variant_order'])

## Comparison of Test vs. Generalization Runtimes per Project + Variant  

In [ ]:
query = """
SELECT 
    rc.project_id,
    rc.project_name,
    rc.variant,
    avg(rc.t_runtime * 1000) AS mean_t_runtime_ms,
    avg(rc.g_runtime * 1000) AS mean_g_runtime_ms,
    avg(rc.runtime_diff * 1000) AS mean_runtime_diff_ms,
    avg(rc.g_runtime) / avg(rc.t_runtime) AS ratio_of_mean_runtimes,
    min(tries) AS tries,
    avg(rc.runtime_diff_per_try * 1000) AS mean_runtime_diff_per_try_ms
FROM mv_runtime_comparison_test_vs_generalization rc
GROUP BY rc.project_id, rc.project_name, rc.variant, rc.variant_order
ORDER BY rc.project_id, rc.project_name, rc.variant_order
"""

df = pd.read_sql_query(query, conn)
display(df.loc[:, df.columns != 'variant_order'])

## Comparison of Test vs. Generalization Runtimes per Variant  

In [ ]:
query = """
SELECT 
    rc.variant,
    avg(rc.t_runtime * 1000) AS mean_t_runtime_ms,
    avg(rc.g_runtime * 1000) AS mean_g_runtime_ms,
    avg(rc.runtime_diff * 1000) AS mean_runtime_diff_ms,
    avg(rc.g_runtime) / avg(rc.t_runtime) AS ratio_of_mean_runtimes,
    min(tries) AS tries,
    avg(rc.runtime_diff_per_try * 1000) AS mean_runtime_diff_per_try_ms
FROM mv_runtime_comparison_test_vs_generalization rc
GROUP BY rc.variant, rc.variant_order
ORDER BY rc.variant_order
"""

df = pd.read_sql_query(query, conn)
display(df.loc[:, df.columns != 'variant_order'])


## Comparison of Test vs. Generalization Runtimes per Variant (Plot)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style for the plot
plt.style.use('ggplot')
sns.set_palette("colorblind")

# Create figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Extract data for plotting
variants = df['variant']
mean_diff = df['mean_runtime_diff_ms']
mean_diff_per_try = df['mean_runtime_diff_per_try_ms']

# Plot 1: Mean Runtime Difference
bars1 = ax1.bar(variants, mean_diff, color=sns.color_palette("colorblind")[0], alpha=0.8)

# Add a horizontal line at y=0 for first plot
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Add data labels on top of each bar for first plot
for bar in bars1:
    height = bar.get_height()
    if height >= 0:
        va = 'bottom'
        offset = 5  # Increased offset for positive values
    else:
        va = 'top'
        offset = -5  # Increased offset for negative values
    ax1.text(bar.get_x() + bar.get_width()/2., height + offset,
            f'{height:.2f}', ha='center', va=va)

# Customize the first plot
ax1.set_title('Mean Runtime Difference (ms)', fontsize=14, pad=20)
ax1.set_xlabel('Variant', fontsize=12)
ax1.set_ylabel('Mean Runtime Difference (ms)\n(Generalization - Test)', fontsize=12)
ax1.grid(axis='y', linestyle='--', alpha=0.7)
ax1.tick_params(axis='x', rotation=45, labelsize=10)

# Plot 2: Mean Runtime Difference Per Try
bars2 = ax2.bar(variants, mean_diff_per_try, color=sns.color_palette("colorblind")[1], alpha=0.8)

# Add a horizontal line at y=0 for second plot
ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)

# Add data labels on top of each bar for second plot
for bar in bars2:
    height = bar.get_height()
    if height >= 0:
        va = 'bottom'
        offset = 5  # Increased offset for positive values
    else:
        va = 'top'
        offset = -5  # Increased offset for negative values
    ax2.text(bar.get_x() + bar.get_width()/2., height + offset,
            f'{height:.2f}', ha='center', va=va)

# Customize the second plot
ax2.set_title('Mean Runtime Difference Per Try (ms)', fontsize=14, pad=20)
ax2.set_xlabel('Variant', fontsize=12)
ax2.set_ylabel('Mean Runtime Difference Per Try (ms)', fontsize=12)
ax2.grid(axis='y', linestyle='--', alpha=0.7)
ax2.tick_params(axis='x', rotation=45, labelsize=10)

# Adjust y-axis limits to add padding for labels
y1_min, y1_max = ax1.get_ylim()
y2_min, y2_max = ax2.get_ylim()
ax1.set_ylim(y1_min - abs(y1_min*0.1), y1_max + abs(y1_max*0.1))
ax2.set_ylim(y2_min - abs(y2_min*0.1), y2_max + abs(y2_max*0.1))

# Add a main title for the entire figure
fig.suptitle('Runtime Comparison: Test vs. Generalization', fontsize=16)

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()
